# SPK-2 — Executor / storage memory pressure & OOM

**Break → Detect → Fix → Prove.** Over-caching a frame you don't reuse, plus too-few shuffle
partitions, makes a heavy aggregation fight for memory → heavy **GC**, **spill**, and on a small
enough box a real **OutOfMemory** (container exit 137).

**Pre-requisite:** `make up` → Spark UI at http://localhost:4040. For a *real* OOM, use
`make up-constrained` (2 GB box, 1 GB driver).

**Laptop-safety / why we describe the hard OOM:** a genuine OOM over **Spark Connect**
**hard-kills the session** (every later cell then raises `[NO_ACTIVE_SESSION]`). So we **describe**
the OOM antipattern (+ recovery) and **execute a contained version** — the same memory pressure as
**spill / GC**, which is visible and reversible — then the fix.

In [ ]:
from common.spark_session import spark, display_df, reconnect
from common.profiles import apply_profile, profile_summary
from common.datagen import wide_rows
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F
print("Spark UI:", "http://localhost:4040")
spark

## Step 0 — Parameters & a wide frame

Wide rows = large bytes per row, so caching them and aggregating them both stress memory.

In [ ]:
N_ROWS = 8_000_000
N_COLS = 30
GROUPS = 1_000

wide = (wide_rows(spark, n_rows=N_ROWS, n_cols=N_COLS)
        .withColumn("k", F.pmod(F.col("row_id"), F.lit(GROUPS))))   # a key to aggregate by
sum_cols = [F.sum(f"c{i}").alias(f"s{i}") for i in range(N_COLS)]   # a heavy, wide aggregation
print(f"wide: ~{N_ROWS:,} rows x {N_COLS} cols (lazy), aggregated into {GROUPS:,} groups")

## 1. The antipattern — over-cache + too-few partitions (described, not run)

```python
apply_profile(spark, "constrained")     # 16 shuffle partitions
wide.cache(); wide.count()              # pins the whole wide frame in STORAGE memory...
wide.groupBy("k").agg(*sum_cols).count()  # ...while a heavy agg needs EXECUTION memory -> contention
```

Pinning a big frame in storage memory leaves little for execution; with only 16 partitions each
task's working set is huge → relentless **GC**, then **OOM**. Over Spark Connect a real OOM
**kills the session**. We don't run that here; instead we run the *contained* version below (it
spills instead of dying), which shows the same pressure safely.

## Recovery — if you *do* OOM the session

```python
spark = reconnect()        # fresh Spark Connect session after a driver/executor death
```

> **Gotcha:** rebuild any DataFrames created before the death — they're bound to the dead session.

## 2. Detect it — read the Spark UI

http://localhost:4040 (see [`docs/spark-ui-guide.md`](../../docs/spark-ui-guide.md)):
- **Executors** tab → **GC Time** rivaling Task Time = memory pressure; killed executors / `exit 137` = OOM.
- **Stages → Tasks** → **Spill (memory/disk)** columns.
- **Storage** tab → how much is pinned by `cache()`.

## Break it (contained) — heavy wide aggregation under tight memory

Same pressure as the antipattern, but **without** over-caching, so it **spills** instead of dying.

In [ ]:
apply_profile(spark, "constrained")     # AQE off, shuffle.partitions = 16
profile_summary(spark)
m_tight = measure(spark, "16 partitions (tight)", lambda: wide.groupBy("k").agg(*sum_cols).count())
print("tight metrics:", m_tight)

## 3. Fix it — right-size partitions + AQE, and cache only what you reuse

More shuffle partitions shrink each task's working set; AQE coalesces the excess. And **don't**
pin frames you read once — `cache()` only a frame reused many times, and `unpersist()` when done.

In [ ]:
apply_profile(spark, "tuned")           # AQE on, shuffle.partitions = 200 (coalesced at runtime)
m_tuned = measure(spark, "200 partitions + AQE", lambda: wide.groupBy("k").agg(*sum_cols).count())
print("tuned metrics:", m_tuned)

## 4. Prove it

More, smaller partitions (and not over-caching) cut spill and GC — the heavy aggregation fits in
memory and finishes faster.

In [ ]:
compare([m_tight, m_tuned])
print("\nMemory pressure is about working-set size: right-size partitions, and cache only reuse.")

## Takeaways & "in real production…"

- **Don't `cache()` what you read once** — it steals execution memory; `unpersist()` when done.
- **Right-size `shuffle.partitions`** (or let AQE coalesce) so each task's working set fits.
- A real executor/driver OOM over **Spark Connect kills the session** — recover with `reconnect()`
  and rebuild DataFrames; in prod, alert on GC time and `exit 137`, and tune `spark.memory.fraction`.

## Teardown

In [ ]:
spark.catalog.clearCache()
apply_profile(spark, "tuned")
print("Done. Caches cleared, profile reset to 'tuned'.")